# Smart Pole 補值套件 — 範例

情境:使用者在儀表板選了「某根竿體要消失 + K 根鄰居 + 一個時間點」,我們跑 11 個演算法,顯示每個的預測值跟誤差。

**這份 notebook 跑完 ≈ 5 秒(CPU)**。

## 1. 載資料

In [1]:
# 若還沒 `pip install -e .`,先把 parent 目錄加到 sys.path,讓 import 能跑
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

from smart_pole_imputation import load_pole_hourly, impute_single, summarize, list_models

dataset = load_pole_hourly(
    'sample_data/pole_hourly_sample.parquet',
    station_info_path='sample_data/MOENV_iot_station.csv',
)
print(f'T={len(dataset.timestamps)} 小時, S={len(dataset.station_ids)} 站')
print(f'時間範圍 {dataset.timestamps.min()} ~ {dataset.timestamps.max()}')
print(f'有座標的站數 {len(dataset.coords)}')
print(f'可用演算法 ({len(list_models())} 個): {list_models()}')

T=336 小時, S=60 站
時間範圍 2026-01-21 23:00:00 ~ 2026-02-04 23:00:00
有座標的站數 60
可用演算法 (13 個): ['corr_weighted', 'dineof', 'elastic_net', 'gaussian_kernel', 'gls', 'idw', 'idw_optimal', 'kriging', 'mean', 'spatial_gp', 'st_gp', 'timelag_ridge', 'weighted_ridge']


## 2. 使用者選擇

儀表板會傳給後端三個東西:`target_id`、`neighbor_ids`、`target_time`。

In [2]:
target_id     = dataset.station_ids[0]
neighbor_ids  = list(dataset.station_ids[1:6])
target_time   = dataset.timestamps[200]

print(f'要消失的竿體:  {target_id}')
print(f'使用者選的鄰居: {neighbor_ids}')
print(f'時間點:        {target_time}')

要消失的竿體:  7527275539
使用者選的鄰居: ['7527351738', '7527563599', '7526409697', '7526625381', '7527014441']
時間點:        2026-01-30 07:00:00


## 3. 一行跑完所有演算法

In [3]:
results = impute_single(
    dataset,
    target_id=target_id,
    neighbor_ids=neighbor_ids,
    target_time=target_time,
    skip_gpu=True,    # 沒 GPU 就跳 st_gp;有 GPU 改 False
)

df = summarize(results)
df

,model,prediction,ground_truth,absolute_error,train_mae,elapsed_s,error
0,gls,16.251341,16.240281,0.011060,0.638525,0.377329,None
1,weighted_ridge,16.253653,16.240281,0.013371,0.949513,0.644354,None
2,timelag_ridge,16.202823,16.240281,0.037458,0.628366,0.001644,None
3,elastic_net,16.296523,16.240281,0.056241,0.628398,0.004692,None
4,dineof,16.043276,16.240281,0.197005,NaN,0.014570,None
5,kriging,22.250970,16.240281,6.010688,4.299345,0.111093,None
6,gaussian_kernel,22.837311,16.240281,6.597029,6.419483,0.000140,None
7,spatial_gp,22.927892,16.240281,6.687610,5.787110,2.128842,None
8,idw_optimal,23.248301,16.240281,7.008020,6.766306,0.002891,None
9,idw,23.329068,16.240281,7.088787,7.157150,0.000261,None


**儀表板右側框要顯示的資料就是上面這份 DataFrame**。每一列:

| 欄位 | 意義 |
|---|---|
| `prediction` | 該演算法的預測值 (µg/m³) |
| `ground_truth` | 實際真值(從歷史資料抓的) |
| `absolute_error` | `|pred − true|`,儀表板顯示的「MAE」 |
| `train_mae` | 該演算法在訓練集上的典型誤差(`dineof`/`st_gp` 為 NaN) |
| `elapsed_s` | 該演算法 fit+predict 牆鐘時間 |

## 4. 看單一演算法的可解釋性

每個演算法的 `explain` 欄回傳「為什麼這樣預測」。例:`elastic_net` 會吐被 L1 挑出來的 features。

In [4]:
import json
ex = results['elastic_net']['explain']
preview = {
    'alpha': ex['alpha'],
    'l1_ratio': ex['l1_ratio'],
    'intercept': ex['intercept'],
    'n_features': ex['n_features'],
    'n_features_kept': ex['n_features_kept'],
    'top_5_kept': ex['kept_features'][:5],
}
print(json.dumps(preview, indent=2, ensure_ascii=False))

{
  "alpha": 0.05,
  "l1_ratio": 0.5,
  "intercept": 1.5141943751565634,
  "n_features": 50,
  "n_features_kept": 34,
  "top_5_kept": [
    [
      "tgt_t-1",
      0.64344499081841
    ],
    [
      "nb3_t",
      0.17581776181666808
    ],
    [
      "nb2_roll24",
      0.15044252722965648
    ],
    [
      "nb1_t-2",
      -0.1314333832526747
    ],
    [
      "nb2_t-1",
      0.13118088783318294
    ]
  ]
}


In [5]:
ex = results['kriging']['explain']
print(json.dumps({k: ex.get(k) for k in ['variogram_model', 'variogram_params', 'weights_sum', 'n_pairs_used']},
                 indent=2, ensure_ascii=False))

{
  "variogram_model": "exponential",
  "variogram_params": {
    "sill": 1620288.8545129462,
    "range": 360052832.4088335,
    "nugget": 1.7908601704230432e-36
  },
  "weights_sum": 0.9999999999999999,
  "n_pairs_used": 5
}


## 5. 自訂超參或選定演算法

In [6]:
results_subset = impute_single(
    dataset, target_id, neighbor_ids, target_time,
    models=['mean', 'elastic_net'],
    model_params={'elastic_net': {'alpha': 0.5}},
)
summarize(results_subset)

,model,prediction,ground_truth,absolute_error,train_mae,elapsed_s,error
0,elastic_net,16.300761,16.240281,0.06048,0.677361,0.002492,None
1,mean,24.815881,16.240281,8.57560,9.187979,0.000183,None


## 6. 直接用 imputer class(進階)

若需要自己組 fit/predict 流程,可以直接從 `REGISTRY` 拿 class:

In [7]:
from smart_pole_imputation import REGISTRY

ElasticNet = REGISTRY['elastic_net']
print('class:', ElasticNet)
print('tier:', ElasticNet.tier, ' explainability:', ElasticNet.explainability)
print('signature: fit(X, y, meta), predict(X, meta), explain()')

class: <class 'smart_pole_imputation.imputers.ElasticNetImputer'>
tier: 3  explainability: 10
signature: fit(X, y, meta), predict(X, meta), explain()
